# Pre Heating Analysis
Winter

In [ ]:
#import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.ticker as ticker


time_frame = 48 # [hours]
time_hours = np.linspace(0,time_frame,time_frame)
time_mins = np.linspace(0, time_frame * 60, time_frame * 60)

# Weather Curve
outdoor_air_temp = 2 * np.sin((np.pi/(12*60)) * time_mins + np.pi)
upper_80_indoor_temp = (0.31*outdoor_air_temp) + 21.3
lower_80_indoor_temp = (0.31*outdoor_air_temp) + 14.3
indoor_air_temp = (upper_80_indoor_temp + lower_80_indoor_temp) / 2

# Price Curve
price = np.sin(time_mins)

# plot air temps
fig, ax = plt.subplots() 
ax.plot(time_mins, outdoor_air_temp, label='outdoor air temp', color='orange')
ax.plot(time_mins, indoor_air_temp, label='indoor air temp', color='blue')
ax.set_xlabel('Time (mins)')
ax.set_ylabel('temperature (degrees C)')
ax.set_title('Outdoor Air Temperature')
ax.grid(True)

ax.legend()
plt.tight_layout()
plt.show()



In [ ]:
# function definitions

# determineWattsNeededForHuman: calculating watts of output needed for the human and graphing
# human params: met_rate, work_rate, body_cp, body_mass, surface_area_human, thermal flexibility, average_temp_midpoint
# heat transfer params: indoor_air_temp, joules_of_preheating, mins_of_preheating

def calculateWattsNeededForHuman(indoor_air_temp, joules_of_preheating=45000, mins_of_preheating=60, met_rate=65, work_rate=0, body_cp=2980, body_mass=62, surface_area_human=1.8, thermal_flexibility=0.2, average_temp_midpoint=35.2, time_coasting_begins=1400):
    # MANAGE HUMAN PARAMETERS
    mean_skin_temp = (0.3812 * (indoor_air_temp)) + 22.406 # get a more accurate model for skin temp
    mean_core_temp = np.array([36.5] * len(time_mins))
    mean_body_temp = ((0.2*mean_skin_temp) + (0.8*mean_core_temp))

    # calculate joules of preheating needed
    if(joules_of_preheating == None):
        joules_of_preheating = body_cp * body_mass * thermal_flexibility
    
    watts_per_hourly_range = np.empty(6)
    for i in range(len(watts_per_hourly_range)) :
        watts_per_hourly_range[i] = joules_of_preheating / ( (i+1) * 3600 )

    print(f"Body Specific Heat Capacity: {body_cp} \nAverage Body Mass: {body_mass} \nAverage Body Temp Midpoint {average_temp_midpoint} \nSweat Limit: 36.5 \n\nEnergy Needed to Heat Person from midpoint to high range:\n{joules_of_preheating}")
    print(f"\n Watts of Heat that is needed to Pre-Heat the Human from neutral to the upper comfort limit:\n{watts_per_hourly_range}\nThe array index is the number of hours of preheating")
    watts_of_preheating_needed = (joules_of_preheating) / (time_mins * 60)
    
    # HUMAN AND ENVIRONMENT HEAT TRANSFER
    # convective and radiative
    i_clo = 0.61
    r_clo = 0.155*i_clo
    f_clo = 1.20 # trousers and long sleeved shirt [ASHRAE pg 9.8]
    convective_heat_transfer_coefficient = 3.1
    linear_radiative_heat_transfer_coefficient = 4.7
    htc = convective_heat_transfer_coefficient+linear_radiative_heat_transfer_coefficient
    r_c_ht = (mean_skin_temp - indoor_air_temp) / (r_clo + (1/(f_clo*htc)))

    # evaporative
    skin_wettedness = 0.06 # dry skin
    p_sk = np.power(2.718,(77.3450+0.0057*(mean_skin_temp + 273.15)-7235/(mean_skin_temp+273.15)))/(np.power((mean_skin_temp+273.15),8.2))/1000
    p_a = np.array([101.325] * len(time_mins)) # Convert p_a to a NumPy array
    r_evap_clo = 0.020 # evaporative heat transfer resistance of clothing layer
    evap_htc = 2.2 * htc # evaporative heat transfer coefficient [engineering data and measurements]
    evap_ht = (skin_wettedness*(p_sk-p_a)) / (r_evap_clo + (1/(f_clo*evap_htc)))

    # total heat loss from skin
    q_sk = r_c_ht + evap_ht
    q_sk = -q_sk

    # total heat loss from respiration
    q_res = np.array([met_rate * 0.1] * len(time_mins))

    print(f"total heat loss from skin: {q_sk}\ntotal heat loss from respiration: {q_res}")
    storage = (met_rate - work_rate - (q_sk + q_res)) * surface_area_human
    total_heat_transfer = met_rate - work_rate - (q_sk + q_res)
    print(f"total heat transfer: {total_heat_transfer}")

    # CALCULATE WATTS OF HEAT NEEDED FOR HUMAN
    watts_of_heat_needed_no_pre_heating = storage.copy()
    total_heat_loss = (q_sk+q_res) * surface_area_human
    heat_loss_rate_1 = total_heat_loss[time_coasting_begins]

    # Evaluate the conditions right at coasting
    print("Human Body conditions right before coasting:")
    print(f"Total Heat Loss [W]: {(total_heat_loss)[time_coasting_begins]}")
    print(f"Total Heat Stored Without Pre-Heating [W]: {storage[time_coasting_begins]}")

    # Evaluate the conditions at the lowest air temperature
    print("\nHuman Body conditions at the lowest air temp:")
    print(f"Total Heat Loss [W]: {(total_heat_loss)[1800]}")
    print(f"Total Heat Stored Without Pre-Heating [W]: {storage[1800]}\n")

    # State Assumptions
    print("Pre-Heating Model 1 Assumptions\n- Pre-Heating will occur immediately before coasting")
    print(f"- Rate of Heat Loss when coasting is equal to the rate of heat loss at the maximum indoor temperature: {heat_loss_rate_1}\n")

    # sets the wattage needed for 1 hour of preheating
    average_heat_gain_1 = 0
    watts_of_heat_needed_preheating1 = watts_of_heat_needed_no_pre_heating.copy()
    additional_watts_for_preheating = joules_of_preheating/(mins_of_preheating * 60)

    mean_body_temp_1 = mean_body_temp.copy()
    rate_of_body_temp_gain = (additional_watts_for_preheating*body_mass)/body_cp # [C/s]
    rate_of_body_temp_loss = (heat_loss_rate_1*body_mass)/body_cp  # [C/s]
    for mins in range(time_coasting_begins-mins_of_preheating, time_coasting_begins):
        watts_of_heat_needed_preheating1[mins] += additional_watts_for_preheating
        mean_body_temp_1[mins] = (rate_of_body_temp_gain/60) + mean_body_temp_1[mins - 1]
        average_heat_gain_1 += additional_watts_for_preheating

    # calculate coasting time
    average_heat_gain_1 /= 60
    joules_of_excess_heat_stored_1 = average_heat_gain_1 * 3600
    mins_coasting_time_1 = joules_of_excess_heat_stored_1/(heat_loss_rate_1*60)

    # during coasting the watts that need to be provided to the human is zero
    for mins in range (time_coasting_begins, time_coasting_begins + int(mins_coasting_time_1)):
        watts_of_heat_needed_preheating1[mins] = 0
        mean_body_temp_1[mins] = mean_body_temp_1[mins - 1] - (rate_of_body_temp_loss/60)


    # print results
    print("Pre-Heating Model 1:")
    print(f"Pre-Heating Times: {time_coasting_begins-60} to {time_coasting_begins}")
    print(f"Joules of Heat Stored = {joules_of_excess_heat_stored_1}\nWatts of Additional Heat Provided: {additional_watts_for_preheating}\nRate of Heat Loss during Coasting: {heat_loss_rate_1}")
    print(f"Total Coasting Time: {mins_coasting_time_1} mins or  {mins_coasting_time_1/60} hours")
    print(f"Coasting Times: {time_coasting_begins} to {time_coasting_begins + mins_coasting_time_1}")

    
    #PLOTTING
    plt.rcParams['figure.figsize'] = [10, 5]
    
    fig, ax = plt.subplots() 
    ax.plot(time_mins, watts_of_heat_needed_preheating1, label='Watts of heat provided with 1 hour of pre-heating', color='purple')
    plt.axvspan(time_coasting_begins - 60, time_coasting_begins, color = 'red', label='Pre-Heating Period', alpha = 0.2)
    plt.axvspan(time_coasting_begins, time_coasting_begins+mins_coasting_time_1, color = 'blue', label='coasting period', alpha = 0.2)

    # plot comments
    plt.figtext(
        0.5, 0.01, 
        f"1 Hour of Pre-Heating, Pre-Heating Rate: 12W, Coasting Time: {np.round(mins_coasting_time_1)} mins", 
        wrap=True, 
        horizontalalignment='center', 
        fontsize=9, 
        style='italic'
    )

    # plot settings
    ax_temp = ax.twinx()
    ax_temp.set_ylabel('Temperature (C)')
    ax_temp.plot(time_mins, indoor_air_temp, color='black', linewidth=2, linestyle='--', label='Indoor Air Temperature')
    ax_temp.plot(time_mins, mean_body_temp_1, color='green', linewidth=2, linestyle='--', label='Mean Body Temperature (MBT)')
    plt.axhspan(average_temp_midpoint - thermal_flexibility, average_temp_midpoint + thermal_flexibility, color = 'green', label='comfortable MBT range', alpha = 0.1)

    ax_temp.tick_params(axis='y', labelcolor='black')
    ax.xaxis.set_major_locator(ticker.MultipleLocator(720))
    ax.xaxis.set_minor_locator(ticker.MultipleLocator(60))
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax_temp.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc='lower right')

    ax.set_xlabel('Time (mins)')
    ax.set_ylabel('Heat (W)')
    ax.set_title('Watts of Heat Needed for 1 Hour of Pre-Heating and the corresponding coasting time')
    ax.grid(True)

    plt.xlim(1300,1540)
    plt.tight_layout()
    plt.show()

calculateWattsNeededForHuman(indoor_air_temp)
   


## No Preheating
brainstorming: Preheating as an integer - when preheating = 0, no preheating, integer value = amount of time of preheating.

In [ ]:
# Storage 

# Occupant Variables
met_rate = 65
work_rate = 0
body_cp = 3500
body_mass = 62
average_temp_midpoint = 35.2
surface_area_human = 1.8

mean_skin_temp = (0.3812 * (indoor_air_temp)) + 22.406 # get a more accurate model for skin temp
mean_core_temp = np.array([36.5] * len(time_mins))
mean_body_temp = ((0.2*mean_skin_temp) + (0.8*mean_core_temp))

thermal_flexibility_mean_body_temp = 0.2 # the human body can comfortably increase temperature by about 0.2 degrees before physiological temperature regulation kicks in.
energy_needed = body_cp * body_mass * thermal_flexibility_mean_body_temp
watts_per_hourly_range = np.empty(6)

for i in range(len(watts_per_hourly_range)) :
    watts_per_hourly_range[i] = energy_needed / ( (i+1) * 3600 )

print(f"Body Specific Heat Capacity: {body_cp} \nAverage Body Mass: {body_mass} \nAverage Body Temp Midpoint {average_temp_midpoint} \nSweat Limit: 36.5 \n\nEnergy Needed to Heat Person from midpoint to high range:\n{energy_needed}")
print(f"\n Watts of Heat that is needed to Pre-Heat the Human from neutral to the upper comfort limit:\n{watts_per_hourly_range}\nThe array index is the number of hours of preheating")
watts_of_preheating_needed = (energy_needed) / (time_mins * 60)
# plot change in watts needed to preheat over the preheating time period
fig, ax = plt.subplots()
ax.plot(time_mins, watts_of_preheating_needed, label='mean body temp', color='red')

ax.set_xlabel('Time (mins)')
ax.set_ylabel('Watts Needed (W)')
ax.set_title('Watts needed to preheat vs. mins of preheating condition')
ax.grid(True)

ax.legend()
plt.xlim(0,180)
plt.tight_layout()
plt.show()

# plot mean body temperature range
fig, ax = plt.subplots()
ax.plot(time_mins, mean_body_temp, label='mean body temp', color='green')
ax.plot(time_mins, mean_body_temp + 0.2, label='mean body temp - upper comfort limit', color='red')
ax.plot(time_mins, mean_body_temp - 0.2, label='mean body temp - lower comfort limit', color='blue')

ax.set_xlabel('Time (mins)')
ax.set_ylabel('Temp (C)')
ax.set_title('Mean Body Temperature Over Time with Higher and Lower Comfort Limits')
ax.grid(True)

ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Winter - No Pre-Heating Model

# Heat Balance Analysis

# HEAT LOSS VIA SKIN

# convective and radiative
i_clo = 0.61
r_clo = 0.155*i_clo
f_clo = 1.20 # trousers and long sleeved shirt [ASHRAE pg 9.8]
convective_heat_transfer_coefficient = 3.1
linear_radiative_heat_transfer_coefficient = 4.7
htc = convective_heat_transfer_coefficient+linear_radiative_heat_transfer_coefficient
r_c_ht = (mean_skin_temp - indoor_air_temp) / (r_clo + (1/(f_clo*htc)))

# evaporative
skin_wettedness = 0.06 # dry skin
p_sk = np.power(2.718,(77.3450+0.0057*(mean_skin_temp + 273.15)-7235/(mean_skin_temp+273.15)))/(np.power((mean_skin_temp+273.15),8.2))/1000
p_a = np.array([101.325] * len(time_mins)) # Convert p_a to a NumPy array
r_evap_clo = 0.020 # evaporative heat transfer resistance of clothing layer
evap_htc = 2.2 * htc # evaporative heat transfer coefficient [engineering data and measurements]
evap_ht = (skin_wettedness*(p_sk-p_a)) / (r_evap_clo + (1/(f_clo*evap_htc)))

# total heat loss from skin
q_sk = r_c_ht + evap_ht
q_sk = -q_sk

# HEAT LOSS VIA RESPIRATION
c_res = 0.0014 * met_rate * (34 - (indoor_air_temp+273.15))
e_res = 0.0173 * met_rate * (5.87 - p_a)
#e_res = np.array([15] * len(time_hours))
#print(f"c_res: {c_res}")
#print(f"e_res: {e_res}")
#q_res = c_res + e_res
q_res = np.array([met_rate * 0.1] * len(time_mins))

print(f"total heat loss from skin: {q_sk}\ntotal heat loss from respiration: {q_res}")
storage = met_rate - (q_sk + q_res)
#print(f"Storage:{storage}")

total_heat_transfer = met_rate - (q_sk + q_res)
print(f"total heat transfer: {total_heat_transfer}")

# graphing again!
fig, ax = plt.subplots() 
ax.plot(time_mins, storage, label='storage', color='orange')
ax.plot(time_mins, (q_sk+q_res), label='total heat lost', color='blue')
#ax.plot(time_hours, q_sk, label='heat loss from skin', color='blue')
#ax.plot(time_hours, q_res, label='heat loss from respiration', color='green')
ax.plot(time_mins, np.array([met_rate] * len(time_mins)), label='heat from met', color='red')

ax.set_xlabel('Time (mins)')
ax.set_ylabel('Heat Flux (W/m^2))')
ax.set_title('Heat stored and Transfered via Respiration and Skin')
ax.grid(True)

ax.legend()
plt.xlim(0,3000)
plt.tight_layout()
plt.show()


In [ ]:
# No Pre-Heating Conclusions
# To achieve comfort with zero pre-heating, the human body should store a minimal amount of heat - as a result, the watts of heat that need to be inputted into the human body to 
# maintain comfort is equivalent to the storage.

watts_of_heat_needed_no_pre_heating = storage * surface_area_human
print(watts_of_heat_needed_no_pre_heating)

# Plot components - indoor air temp, heat loss, heat storage, watts inputted
fig, ax = plt.subplots() 
#ax.plot(time_mins, storage * surface_area_human, label='Heat Stored in the Human Body', color='orange')
#ax.plot(time_mins, (q_sk+q_res) * surface_area_human, label='Total Heat Lost from the Human Body', color='green')
ax.plot(time_mins, watts_of_heat_needed_no_pre_heating, label='watts of heated needed for thermal comfort - no preheating', color='red')

ax.set_xlabel('Time (mins)')
ax.set_ylabel('Heat (W)')
ax.set_title('Heat Transfer for Thermal Comfort - Non-Pre-Heating Condition')
ax.grid(True)

ax.legend()
plt.xlim(0,3000)
plt.tight_layout()
plt.show()


### Non Pre-Heating Condition Conclusions
The Human body is at optimal thermal comfort with thermal storage is closest to zero and is within +- 15 W [from comfortable heat analysis]
In the non-preheating condition, optimal thermal comfort is achieved by making the amount of heat provided to the human body equivalent to the watts of heat being stored by the human body. In this winter scenario,
the human body is storing large amounts of heat because the indoor temperture is lower. By providing an equivalent amount of heat, the human will stay at an optimal thermal comfort level.

## Pre-Heating Condition
Within the Pre=heating condition, we expand the range of thermal storage allowed - thermal storage within the human body is allowed to vary within a comfortable range defined by how much the mean body temperature can
comfortable change. According to the thermal storage analysis, if we have a flexibility range of about 0.2 C, then the human body can store an excess of about 43 kJ of heat. These 43kJ can  be distributed over a range of time depending on 1. how much time will be spent preheating and 2. how much time will be spent coasting.

In our analysis of the pre-heating condition, the amount of time necessary for coasting will be defined and the amount of time needed for pre-heating will be determined from there.

Pre-Heating hours will be determined by the coldest hours of the day, when the temperature is still actively decreasing. (indoor_air_temperature curve is concave up)

In [ ]:
# Highest Demand from 1440-2160
# Pre-heat from 720 - 1440
# Preheating Condition 1 - 1 Hour of Coasting Time ----------------------------------------------------------------------------------------
total_heat_loss = q_sk+q_res
time_coasting_begins = 1440 # [time in mins]
heat_loss_rate_1 = total_heat_loss[time_coasting_begins] * surface_area_human
plt.rcParams['figure.figsize'] = [10, 5]

# Evaluate the conditions right at coasting
print("Human Body conditions right before coasting:")
print(f"Total Heat Loss [W]: {(total_heat_loss)[time_coasting_begins] * surface_area_human}")
print(f"Total Heat Stored Without Pre-Heating [W]: {storage[time_coasting_begins]}")

# Evaluate the conditions at the lowest air temperature
print("\nHuman Body conditions at the lowest air temp:")
print(f"Total Heat Loss [W]: {(total_heat_loss)[1800] * surface_area_human}")
print(f"Total Heat Stored Without Pre-Heating [W]: {storage[1800]}\n")

# State Assumptions
print("Pre-Heating Model 1 Assumptions\n- Pre-Heating will occur immediately before coasting")
print(f"- Rate of Heat Loss when coasting is equal to the rate of heat loss at the maximum indoor temperature: {heat_loss_rate_1}\n")

# sets the wattage needed for 1 hour of preheating
average_heat_gain_1 = 0
watts_of_heat_needed_preheating1 = watts_of_heat_needed_no_pre_heating.copy()

mean_body_temp_1 = mean_body_temp.copy()
rate_of_body_temp_gain = (12*body_mass)/body_cp # [C/s]
rate_of_body_temp_loss = (heat_loss_rate_1*body_mass)/body_cp  # [C/s]
for mins in range(time_coasting_begins-60, time_coasting_begins):
    watts_of_heat_needed_preheating1[mins] += 12
    mean_body_temp_1[mins] = (rate_of_body_temp_gain/60) + mean_body_temp_1[mins - 1]
    average_heat_gain_1 +=12

# calculate coasting time
average_heat_gain_1 /= 60
joules_of_excess_heat_stored_1 = average_heat_gain_1 * 3600
mins_coasting_time_1 = joules_of_excess_heat_stored_1/(heat_loss_rate_1*60)

# during coasting the watts that need to be provided to the human is zero
for mins in range (time_coasting_begins, time_coasting_begins + int(mins_coasting_time_1)):
    watts_of_heat_needed_preheating1[mins] = 0
    mean_body_temp_1[mins] = mean_body_temp_1[mins - 1] - (rate_of_body_temp_loss/60)


# print and plot conclusions
print("Pre-Heating Model 1:")
print(f"Pre-Heating Times: {time_coasting_begins-60} to {time_coasting_begins}")
print(f"Joules of Heat Stored = {joules_of_excess_heat_stored_1}\nRate of Heat Loss during Coasting: {heat_loss_rate_1}")
print(f"Total Coasting Time: {mins_coasting_time_1} mins or  {mins_coasting_time_1/60} hours")
print(f"Coasting Times: {time_coasting_begins} to {time_coasting_begins + mins_coasting_time_1}")

# plot the PRE-HEATING 1 Model
fig, ax = plt.subplots() 
#ax.plot(time_mins, watts_of_heat_needed_no_pre_heating, label='watts of heat needed with NO-PREHEATING', color='orange')
ax.plot(time_mins, watts_of_heat_needed_preheating1, label='Watts of heat provided with 1 hour of pre-heating', color='purple')
plt.axvspan(time_coasting_begins - 60, time_coasting_begins, color = 'red', label='Pre-Heating Period', alpha = 0.2)
plt.axvspan(time_coasting_begins, time_coasting_begins+mins_coasting_time_1, color = 'blue', label='coasting period', alpha = 0.2)


# plot comments
plt.figtext(
    0.5, 0.01, 
    f"1 Hour of Pre-Heating, Pre-Heating Rate: 12W, Coasting Time: {np.round(mins_coasting_time_1)} mins", 
    wrap=True, 
    horizontalalignment='center', 
    fontsize=9, 
    style='italic'
)

# plot settings
ax_temp = ax.twinx()
ax_temp.set_ylabel('Temperature (C)')
ax_temp.plot(time_mins, indoor_air_temp, color='black', linewidth=2, linestyle='--', label='Indoor Air Temperature')
ax_temp.plot(time_mins, mean_body_temp_1, color='green', linewidth=2, linestyle='--', label='Mean Body Temperature (MBT)')
plt.axhspan(average_temp_midpoint - thermal_flexibility_mean_body_temp, average_temp_midpoint + thermal_flexibility_mean_body_temp, color = 'green', label='comfortable MBT range', alpha = 0.1)

ax_temp.tick_params(axis='y', labelcolor='black')
ax.xaxis.set_major_locator(ticker.MultipleLocator(720))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(60))
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax_temp.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='lower right')

ax.set_xlabel('Time (mins)')
ax.set_ylabel('Heat (W)')
ax.set_title('Watts of Heat Needed for 1 Hour of Pre-Heating and the corresponding coasting time')
ax.grid(True)

plt.xlim(1300,1540)
plt.tight_layout()
plt.show()

# PLOT THE NO-PREHEATING CONDITION ------------------------------------------------------------------------------------------------------------
average_heat_gain_0 = 0
heat_loss_rate_0=heat_loss_rate_1.copy()
for mins in range(time_coasting_begins-60, time_coasting_begins):
    #average_heat_gain_0 += watts_of_heat_needed_no_pre_heating[mins]
    average_heat_gain_0 =0

# calculate coasting time
average_heat_gain_0 /= 60
joules_of_excess_heat_stored_0 = average_heat_gain_0 * 3600
mins_coasting_time_0 = joules_of_excess_heat_stored_0/(heat_loss_rate_0*60)

# print and plot conclusions
print("\nNo Pre-Heating Model 1:")
#print(f"Pre-Heating Times: {time_coasting_begins-60} to {time_coasting_begins}")
print(f"Joules of Heat Stored = {joules_of_excess_heat_stored_0}\nRate of Heat Loss during Coasting: {heat_loss_rate_0}")
print(f"Total Coasting Time: {mins_coasting_time_0} mins or  {mins_coasting_time_0/60} hours")
print(f"Coasting Times: {time_coasting_begins} to {time_coasting_begins + mins_coasting_time_0}")

fig, ax = plt.subplots() 
ax.plot(time_mins, watts_of_heat_needed_no_pre_heating, label='watts of heat needed with NO-PREHEATING', color='orange')
plt.axvspan(time_coasting_begins, time_coasting_begins+mins_coasting_time_0, color = 'blue', label='comfortable body temperature range', alpha = 0.2)

# plot comments
plt.figtext(
    0.5, 0.01, 
    "No Preheating, indoor air temp relates to the right hand axis", 
    wrap=True, 
    horizontalalignment='center', 
    fontsize=9, 
    style='italic'
)

# plot settings
ax_temp = ax.twinx()
ax_temp.set_ylabel('Indoor Air Temperature (C)')
ax_temp.plot(time_mins, indoor_air_temp, color='black', linewidth=2, linestyle='--', label='Indoor Air Temperature')
ax_temp.plot(time_mins, mean_body_temp, color='green', linewidth=2, linestyle='--', label='Mean Body Temperature (MBT)')
plt.axhspan(average_temp_midpoint - thermal_flexibility_mean_body_temp, average_temp_midpoint + thermal_flexibility_mean_body_temp, color = 'green', label='comfortable MBT range', alpha = 0.1)
ax_temp.tick_params(axis='y', labelcolor='black')
ax.xaxis.set_major_locator(ticker.MultipleLocator(720))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(60))
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax_temp.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='lower right')

ax.set_xlabel('Time (mins)')
ax.set_ylabel('Heat (W)')
ax.set_title('Watts of Heat Needed for 1 Hour of Pre-Heating and the corresponding coasting time')
ax.grid(True)

plt.xlim(1300,1500)
plt.tight_layout()
plt.show()

# SHOW DIFFERENCE BETWEEN NO PRE-HEATING AND PRE-HEATING
print(f"Difference between coasting with or without pre-heating: {mins_coasting_time_1-mins_coasting_time_0} mins or {(mins_coasting_time_1-mins_coasting_time_0) / 60} hours")


## Pre-Heating, Coasting, and the Limits of Comfort

In [ ]:
# Finding Pre-Heating Time/ Goal for Joules Stored based on pre-determined coasting time.

# Calculate the Joules that need to be stored based on the amount of coasting time needed
joules_heat_stored = np.zeros(len(time_mins))
coasting_body_temperatures = np.zeros(len(time_mins))
for mins_coasting in range(len(time_mins)):
    joules_heat_stored[mins_coasting] = heat_loss_rate_0 * mins_coasting * 60
    coasting_body_temperatures[mins_coasting] = (joules_heat_stored[mins_coasting]/(body_mass*body_cp)) + average_temp_midpoint

# Print output
print(f"Joules of Heat that need to be stored depending on the amount of coasting time wanted: {joules_heat_stored}") # linear relationship
print(f"Mean Body Temperature When Human is Preheating: {coasting_body_temperatures}")
print(f"Mean Body Temperature increase: {coasting_body_temperatures-average_temp_midpoint}")
print(f"Difference from the comfortable net MBT range: {0.2 - coasting_body_temperatures}")

# Plot Outcomes
fig, ax = plt.subplots() 
ax.plot(time_mins, coasting_body_temperatures, label='body temperature for time of coasting', color='blue')
plt.axhspan(average_temp_midpoint - thermal_flexibility_mean_body_temp, average_temp_midpoint + thermal_flexibility_mean_body_temp, color = 'brown', label='Mean Body Temperature Comfort Zone', alpha=0.5)
plt.axhspan(average_temp_midpoint - 0.5, average_temp_midpoint + 0.5, color = 'orange', label='Mean Body Temperature Comfort Zone', alpha=0.5)

# plot settings
ax.legend()
ax.xaxis.set_major_locator(ticker.MultipleLocator(120))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(20))

ax.set_xlabel('Time (mins)')
ax.set_ylabel('Heat (W)')
ax.set_title('Coasting Time, Mean Body Temperature (MBT) associated with')
ax.grid(True)

plt.xlim(0,60)
plt.ylim(30,40)
plt.tight_layout()
plt.show()


# Spring Pre-Heating
Average Outdoor Air Temperature: 15 C with a range of 2 C

In [ ]:
# Spring Pre-Heating

# Weather Curve
outdoor_air_temp += 15
upper_80_indoor_temp = (0.31*outdoor_air_temp) + 21.3
lower_80_indoor_temp = (0.31*outdoor_air_temp) + 14.3
indoor_air_temp = (upper_80_indoor_temp + lower_80_indoor_temp) / 2

# Price Curve
price = np.sin(time_mins)

# plot air temps
fig, ax = plt.subplots() 
ax.plot(time_mins, outdoor_air_temp, label='outdoor air temp', color='orange')
ax.set_xlabel('Time (mins)')
ax.set_ylabel('temperature (degrees C)')
ax.set_title('Outdoor Air Temperature')
ax.grid(True)

ax.legend()
plt.tight_layout()
plt.show()

# convective and radiative
i_clo = 0.61
r_clo = 0.155*i_clo
f_clo = 1.20 # trousers and long sleeved shirt [ASHRAE pg 9.8]
convective_heat_transfer_coefficient = 3.1
linear_radiative_heat_transfer_coefficient = 4.7
htc = convective_heat_transfer_coefficient+linear_radiative_heat_transfer_coefficient
r_c_ht = (mean_skin_temp - indoor_air_temp) / (r_clo + (1/(f_clo*htc)))

# evaporative
skin_wettedness = 0.06 # dry skin
p_sk = np.power(2.718,(77.3450+0.0057*(mean_skin_temp + 273.15)-7235/(mean_skin_temp+273.15)))/(np.power((mean_skin_temp+273.15),8.2))/1000
p_a = np.array([101.325] * len(time_mins)) # Convert p_a to a NumPy array
r_evap_clo = 0.020 # evaporative heat transfer resistance of clothing layer
evap_htc = 2.2 * htc # evaporative heat transfer coefficient [engineering data and measurements]
evap_ht = (skin_wettedness*(p_sk-p_a)) / (r_evap_clo + (1/(f_clo*evap_htc)))

# total heat loss from skin
q_sk = r_c_ht + evap_ht
q_sk = -q_sk

# HEAT LOSS VIA RESPIRATION
c_res = 0.0014 * met_rate * (34 - (indoor_air_temp+273.15))
e_res = 0.0173 * met_rate * (5.87 - p_a)
#e_res = np.array([15] * len(time_hours))
#print(f"c_res: {c_res}")
#print(f"e_res: {e_res}")
#q_res = c_res + e_res
q_res = np.array([met_rate * 0.1] * len(time_mins))

print(f"total heat loss from skin: {q_sk}\ntotal heat loss from respiration: {q_res}")
storage = met_rate - (q_sk + q_res)
#print(f"Storage:{storage}")

total_heat_transfer = met_rate - (q_sk + q_res)
print(f"total heat transfer: {total_heat_transfer}")

# Calculate Watts of Heat Needed for No Pre Heating Conditions
watts_of_heat_needed_no_pre_heating = storage * surface_area_human
print(watts_of_heat_needed_no_pre_heating)

# Plot components - indoor air temp, heat loss, heat storage, watts inputted
fig, ax = plt.subplots() 
#ax.plot(time_mins, storage * surface_area_human, label='Heat Stored in the Human Body', color='orange')
#ax.plot(time_mins, (q_sk+q_res) * surface_area_human, label='Total Heat Lost from the Human Body', color='green')
ax.plot(time_mins, watts_of_heat_needed_no_pre_heating, label='watts of heated needed for thermal comfort - no preheating', color='red')

ax.set_xlabel('Time (mins)')
ax.set_ylabel('Heat (W)')
ax.set_title('Heat Transfer for Thermal Comfort - Non-Pre-Heating Condition')
ax.grid(True)

ax.legend()
plt.xlim(0,3000)
plt.tight_layout()
plt.show()